In [1]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT = "payflow-analytics-513907"
DATASET = "payflow_analytics"
client  = bigquery.Client(project=PROJECT)

def bq(sql):
    return client.query(sql).to_dataframe()

def tbl(name):
    return f"`{PROJECT}.{DATASET}.{name}`"

## Load star schema tables

In [2]:
fct   = bq(f"SELECT * FROM {tbl('fct_transactions')}")
dim_m = bq(f"SELECT * FROM {tbl('dim_merchants')}")
dim_d = bq(f"SELECT * FROM {tbl('dim_date')}")
dim_a = bq(f"SELECT * FROM {tbl('dim_accounts')}")

# Enrich fact with merchant and date attributes for analysis
df = (
    fct
    .merge(dim_m, on="merchant_clean", how="left")
    .merge(dim_d, on="transaction_date", how="left")
)

print(f"fct_transactions : {len(fct):,} rows")
print(f"dim_merchants    : {len(dim_m):,} rows")
print(f"dim_date         : {len(dim_d):,} rows")
print(f"dim_accounts     : {len(dim_a):,} rows")
print(f"Fraud rate       : {df['is_fraud'].mean():.2%}")

fct_transactions : 100,133 rows
dim_merchants    : 10 rows
dim_date         : 366 rows
dim_accounts     : 3,133 rows
Fraud rate       : 19.80%


## Overview

In [3]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
transaction_id,100133,100133,57855799-d64e-47f6-8a33-058b55235912,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
account_id,100133,3133,414679ac-7907-4d5f-af68-4428f84b0fbb,116,NaN,NaN,NaN,NaN,NaN,NaN,NaN
merchant_clean,100133,10,Shell,10267,NaN,NaN,NaN,NaN,NaN,NaN,NaN
transaction_date,100133,NaN,NaN,NaN,2026-05-09 14:19:57.711044,2025-11-07 00:00:00,2026-02-08 00:00:00,2026-05-10 00:00:00,2026-08-09 00:00:00,2026-11-07 00:00:00,NaN
hour_of_day,100133.0,<NA>,<NA>,<NA>,9.416246,0.0,4.0,9.0,14.0,23.0,5.731918
currency,100133,5,MXN,20144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
account_country,100133,1,CR,100133,NaN,NaN,NaN,NaN,NaN,NaN,NaN
account_type,100133,3,personal,69363,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,100133.0,NaN,NaN,NaN,170.363525,0.35,16.32,63.8,167.55,15015.63,419.86523
amount_usd,100133.0,NaN,NaN,NaN,113.771902,0.000963,1.05,13.2557,96.9,15219.970254,373.692402


## Fraud rate trend

In [4]:
daily = (
    df.groupby("transaction_date")
    .agg(
        total=("transaction_id", "count"),
        fraud=("is_fraud", "sum"),
        total_usd=("amount_usd", "sum"),
    )
    .assign(fraud_rate=lambda x: x["fraud"] / x["total"])
    .reset_index()
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=daily["transaction_date"], y=daily["total"], name="Transactions", opacity=0.4), secondary_y=False)
fig.add_trace(go.Scatter(x=daily["transaction_date"], y=daily["fraud_rate"], name="Fraud rate", line=dict(color="crimson", width=2)), secondary_y=True)
fig.update_layout(title="Daily transactions and fraud rate", height=400)
fig.update_yaxes(title_text="Transactions", secondary_y=False)
fig.update_yaxes(title_text="Fraud rate", tickformat=".0%", secondary_y=True)
fig.show()

## Amount distribution: fraud vs legit

In [5]:
fig = px.histogram(
    df[df["amount_usd"] < df["amount_usd"].quantile(0.99)],
    x="amount_usd",
    color="is_fraud",
    nbins=80,
    barmode="overlay",
    opacity=0.6,
    color_discrete_map={True: "crimson", False: "steelblue"},
    labels={"amount_usd": "Amount (USD)", "is_fraud": "Fraud"},
    title="Amount distribution (p99 cap)",
)
fig.show()

## Fraud rate by feature

In [6]:
def fraud_rate_bar(col, title):
    agg = (
        df.groupby(col)["is_fraud"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "fraud_rate", "count": "n"})
        .sort_values("fraud_rate", ascending=False)
        .reset_index()
    )
    fig = px.bar(
        agg, x=col, y="fraud_rate", text="n",
        title=title,
        labels={"fraud_rate": "Fraud rate"},
        color="fraud_rate",
        color_continuous_scale="Reds",
    )
    fig.update_yaxes(tickformat=".0%")
    fig.update_traces(texttemplate="%{text:,}", textposition="outside")
    fig.show()

# From fct_transactions
fraud_rate_bar("device_type",  "Fraud rate by device type")
fraud_rate_bar("account_type", "Fraud rate by account type")
fraud_rate_bar("currency",     "Fraud rate by currency")

# From dim_merchants (via join)
fraud_rate_bar("mcc_category", "Fraud rate by MCC category (dim_merchants)")
fraud_rate_bar("risk_tier",    "Fraud rate by risk tier (dim_merchants)")

## Fraud by hour of day

In [7]:
hourly = (
    df.groupby("hour_of_day")["is_fraud"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "fraud_rate", "count": "n"})
    .reset_index()
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=hourly["hour_of_day"], y=hourly["n"], name="Transactions", opacity=0.4), secondary_y=False)
fig.add_trace(go.Scatter(x=hourly["hour_of_day"], y=hourly["fraud_rate"], name="Fraud rate", line=dict(color="crimson", width=2), mode="lines+markers"), secondary_y=True)
fig.update_layout(title="Transactions and fraud rate by hour of day", xaxis_title="Hour", height=400)
fig.update_yaxes(title_text="Transactions", secondary_y=False)
fig.update_yaxes(title_text="Fraud rate", tickformat=".0%", secondary_y=True)
fig.show()

## Fraud rate by quarter (dim_date)

In [8]:
fraud_rate_bar("quarter", "Fraud rate by quarter (dim_date)")

## Fraud patterns

In [9]:
patterns = (
    df[df["is_fraud"]]
    .groupby("fraud_pattern", dropna=False)
    .size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)
patterns["fraud_pattern"] = patterns["fraud_pattern"].fillna("unknown")

fig = px.bar(
    patterns, x="fraud_pattern", y="count",
    title="Fraud transactions by pattern",
    labels={"fraud_pattern": "Pattern", "count": "Transactions"},
    color="count",
    color_continuous_scale="Reds",
)
fig.update_traces(texttemplate="%{y:,}", textposition="outside")
fig.show()

## Numeric feature correlations with fraud

In [10]:
numeric_cols = ["amount_usd", "hour_of_day", "day_of_week", "account_age_days", "latency_ms", "credit_limit"]

corr = (
    df[numeric_cols + ["is_fraud"]]
    .assign(is_fraud=df["is_fraud"].astype(float))
    .corr()
)

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Correlation matrix (numeric features + is_fraud)",
)
fig.show()